*how many listings are there in each neighbourhood?*

*what is the typical number of guests for each type of room?*

In [1]:
import pandas as pd

listings = pd.read_csv('data/listings.csv')
listings.shape

(4574, 84)

## Count per group: value_counts

In [2]:
listings['room_type'].value_counts()

room_type
Entire home/apt    3760
Private room        787
Shared room          24
Hotel room            3
Name: count, dtype: int64

In [3]:
listings['room_type'].value_counts(normalize=True).round(3)


room_type
Entire home/apt    0.822
Private room       0.172
Shared room        0.005
Hotel room         0.001
Name: proportion, dtype: float64

In [4]:
listings['neighbourhood_cleansed'].value_counts()


neighbourhood_cleansed
Södermalms                1145
Norrmalms                  469
Hägersten-Liljeholmens     464
Enskede-Årsta-Vantörs      445
Kungsholmens               420
Östermalms                 297
Bromma                     281
Skarpnäcks                 221
Älvsjö                     214
Farsta                     177
Hässelby-Vällingby         148
Rinkeby-Tensta             108
Skärholmens                 96
Spånga-Tensta               89
Name: count, dtype: int64

In [5]:
listings['bedrooms'].value_counts(dropna=False).sort_index()


bedrooms
0.0      257
1.0     2490
2.0      739
3.0      588
4.0      351
5.0      116
6.0       23
7.0        3
8.0        1
12.0       4
NaN        2
Name: count, dtype: int64

## Group and calculate: groupby

1. Split
2. Calculate
3. Combine


In [6]:
listings.groupby('room_type')['accommodates'].mean().round(2)

room_type
Entire home/apt    3.97
Hotel room         1.33
Private room       1.93
Shared room        4.46
Name: accommodates, dtype: float64

In [7]:
listings.groupby('room_type')['accommodates'].agg(['count', 'mean', 'median', 'max']).round(2)


,count,mean,median,max
room_type,,,,
Entire home/apt,3760,3.97,4.0,16
Hotel room,3,1.33,1.0,2
Private room,787,1.93,2.0,16
Shared room,24,4.46,2.5,14


In [8]:
listings.groupby('room_type')[['accommodates', 'bedrooms', 'beds']].mean().round(2)

,accommodates,bedrooms,beds
room_type,,,
Entire home/apt,3.97,1.87,2.48
Hotel room,1.33,1.00,1.00
Private room,1.93,1.11,1.48
Shared room,4.46,0.79,4.58


In [9]:
listings.groupby('room_type')['price'].mean()

TypeError: dtype 'str' does not support operation 'mean'

In [11]:
summary = listings.groupby('room_type').agg(
    n_listings=('id', 'count'),
    avg_guests=('accommodates', 'mean'),
    median_reviews=('number_of_reviews', 'median')
).round(2)

summary

,n_listings,avg_guests,median_reviews
room_type,,,
Entire home/apt,3760,3.97,8.0
Hotel room,3,1.33,56.0
Private room,787,1.93,12.0
Shared room,24,4.46,81.5


In [12]:
(listings.groupby('neighbourhood_cleansed')['review_scores_rating']
 .agg(['count', 'mean', 'median'])
 .round(2)
 .sort_values('mean')
 )

,count,mean,median
neighbourhood_cleansed,,,
Rinkeby-Tensta,80,4.63,4.80
Skärholmens,76,4.72,4.92
Spånga-Tensta,72,4.77,4.88
Farsta,137,4.78,4.93
Bromma,223,4.79,4.95
Skarpnäcks,196,4.80,4.94
Östermalms,255,4.81,4.90
Kungsholmens,383,4.82,4.88
Södermalms,1058,4.82,4.90


## Two categories at once: crosstab

**How many entire homes, private rooms and other room-types are there in each neighbourhood?**

In [13]:
listings.groupby(['neighbourhood_cleansed', 'room_type']).size().head(8)

neighbourhood_cleansed  room_type      
Bromma                  Entire home/apt    259
                        Private room        22
Enskede-Årsta-Vantörs   Entire home/apt    362
                        Private room        83
Farsta                  Entire home/apt    145
                        Private room        32
Hägersten-Liljeholmens  Entire home/apt    394
                        Private room        70
dtype: int64

In [14]:
pd.crosstab(listings['neighbourhood_cleansed'], listings['room_type'])

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Bromma,259,0,22,0
Enskede-Årsta-Vantörs,362,0,83,0
Farsta,145,0,32,0
Hägersten-Liljeholmens,394,0,70,0
Hässelby-Vällingby,118,0,30,0
Kungsholmens,332,0,80,8
Norrmalms,373,3,86,7
Rinkeby-Tensta,74,0,33,1
Skarpnäcks,167,0,54,0


In [15]:
(pd.crosstab(listings['neighbourhood_cleansed'], listings['room_type'], normalize='index')*100).round(1)

room_type,Entire home/apt,Hotel room,Private room,Shared room
neighbourhood_cleansed,,,,
Bromma,92.2,0.0,7.8,0.0
Enskede-Årsta-Vantörs,81.3,0.0,18.7,0.0
Farsta,81.9,0.0,18.1,0.0
Hägersten-Liljeholmens,84.9,0.0,15.1,0.0
Hässelby-Vällingby,79.7,0.0,20.3,0.0
Kungsholmens,79.0,0.0,19.0,1.9
Norrmalms,79.5,0.6,18.3,1.5
Rinkeby-Tensta,68.5,0.0,30.6,0.9
Skarpnäcks,75.6,0.0,24.4,0.0


## Create new columns: np.where

In [16]:
listings['for_groups'] = listings['accommodates'] >= 4

listings[['accommodates', 'for_groups']].head(6)

,accommodates,for_groups
0,2,False
1,1,False
2,1,False
3,4,True
4,5,True
5,2,False


In [17]:
listings.shape

(4574, 85)

*Copy-on-Write*

In [18]:
guests = listings['accommodates']

guests.iloc[0] = 99

print(guests.iloc[0])
print(listings.loc[0, 'accommodates'])

99
2


In [19]:
import numpy as np

listings['size_type'] = np.where(listings['accommodates'] >= 4, 'group', 'small')

listings['size_type'].value_counts()

size_type
small    2405
group    2169
Name: count, dtype: int64

In [20]:
listings['size_three'] = np.where(
    listings['accommodates'] <= 2, 'small',
    np.where(listings['accommodates'] <= 4, 'medium', 'large')
)

listings['size_three'].value_counts()


size_three
small     1882
medium    1478
large     1214
Name: count, dtype: int64

## Turn numbers intop groups: cut and qcut

In [21]:
pd.cut(listings['accommodates'], bins=[0, 2, 4, 8, 16]).value_counts().sort_index()

accommodates
(0, 2]     1882
(2, 4]     1478
(4, 8]     1117
(8, 16]      97
Name: count, dtype: int64

In [23]:
print(pd.cut(listings['accommodates'], bins=[1, 2, 4, 8, 16]).isna().sum())


392


In [24]:
listings['size_group'] = pd.cut(
    listings['accommodates'],
    bins=[0, 2, 4, 8, 16],
    labels=['1-2', '3-4', '5-8', '9-16']
)

print(listings['size_group'].isna().sum())
listings['size_group'].value_counts().sort_index()

0


size_group
1-2     1882
3-4     1478
5-8     1117
9-16      97
Name: count, dtype: int64

In [25]:
pd.qcut(listings['number_of_reviews'], 4).value_counts().sort_index()

number_of_reviews
(-0.001, 2.0]     1227
(2.0, 9.0]        1083
(9.0, 29.0]       1121
(29.0, 1293.0]    1143
Name: count, dtype: int64

In [26]:
listings['number_of_reviews'].describe()

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [27]:
pd.qcut(listings['availability_365'], 4)


ValueError: Bin edges must be unique: Index([0.0, 0.0, 84.0, 277.0, 365.0], dtype='float64', name='availability_365').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [28]:
print((listings['availability_365']==0).sum())

1494


In [29]:
pd.qcut(listings['availability_365'], 4, duplicates='drop').value_counts().sort_index()


availability_365
(-0.001, 84.0]    2302
(84.0, 277.0]     1131
(277.0, 365.0]    1141
Name: count, dtype: int64

In [30]:
listings['availability_group'] = pd.qcut(
    listings['availability_365'],
    4,
    labels=['low', 'medium', 'high'],
    duplicates='drop'
)

listings['availability_group'].value_counts().sort_index()

availability_group
low       2302
medium    1131
high      1141
Name: count, dtype: int64

## Put it together

**Which neighbourhoods have the highest and the lowest share of listings for groups, with room for four or more guests? And how many listings is each share based on?**

1. Group
2. Calculate
3. Sort

In [31]:
answer = (
    listings.groupby('neighbourhood_cleansed')
    .agg(n_listings=('id', 'count'), share_for_groups=('for_groups', 'mean'))
    .sort_values('share_for_groups', ascending=False)
    .round(3)
)

answer

,n_listings,share_for_groups
neighbourhood_cleansed,,
Bromma,281,0.712
Farsta,177,0.644
Älvsjö,214,0.612
Hässelby-Vällingby,148,0.608
Skarpnäcks,221,0.538
Enskede-Årsta-Vantörs,445,0.526
Hägersten-Liljeholmens,464,0.491
Spånga-Tensta,89,0.461
Kungsholmens,420,0.426
